

# Experimento de Monte Carlo: Consumo de CPU en Modelos ONNX vs RKNN

Este notebook implementa un experimento de Monte Carlo para analizar el consumo de CPU durante el procesamiento de imágenes usando dos tipos de modelos: ONNX y RKNN. Se realizarán 4 experimentos por cada tipo de modelo para obtener estadísticas comparativas robustas.

## Objetivos:|
1. Comparar el rendimiento de CPU entre modelos ONNX y RKNN
2. Analizar la distribución estadística del consumo de recursos
3. Evaluar la variabilidad del rendimiento a través de múltiples experimentos
4. Generar visualizaciones comparativas del uso de CPU por núcleo

## 1. Importar Librerías Requeridas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import cv2
import time
import threading
import random
import math
import os
from glob import glob
from collections import defaultdict
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Librerías específicas para modelos
from ultralytics import YOLO
import onnxruntime as ort
from rknnlite.api import RKNNLite
from text_converter import TextConverter

# Configuración para reproducibilidad
np.random.seed(42)
random.seed(42)

# Configuración de matplotlib
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Librerías importadas correctamente")
print(f"📊 Número de núcleos de CPU detectados: {psutil.cpu_count(logical=True)}")
print(f"💾 RAM total disponible: {psutil.virtual_memory().total / (1024**3):.1f} GB")

## 2. Definir Parámetros del Experimento de Monte Carlo

In [ ]:
# Parámetros del experimento de Monte Carlo
MONTE_CARLO_ITERATIONS = 100  # Número de experimentos por tipo de modelo
CONFIDENCE_LEVEL = 0.95     # Nivel de confianza para intervalos
MONITORING_INTERVAL = 0.1   # Intervalo de monitoreo CPU en segundos
SAMPLE_SIZE_PER_EXP = 3     # Número de imágenes por experimento

# Configuración de directorios
TEST_FOLDERS = ["Testig_Montecarlo/1", "Testig_Montecarlo/2", "Testig_Montecarlo/3", 
                "Testig_Montecarlo/4", "Testig_Montecarlo/5",]

# Configuración de modelos
MODEL_CONFIGS = {
    'ONNX': {
        'yolo_path': "models/best_etiquetas.onnx",
        'ocr_path': 'models/model_ocr.onnx',
        'segrot_path': 'models/model_rotseg.onnx',
        'color': '#FF6B6B',
        'normalization': True  # ONNX requiere normalización ImageNet
    },
    'RKNN': {
        'yolo_path': "best_etiquetas_rknn_model",
        'ocr_path': 'models/model_ocr.rknn',
        'segrot_path': 'models/model_rotseg.rknn',
        'color': '#4ECDC4',
        'normalization': False  # RKNN no requiere normalización manual
    }
}

# Estructura para almacenar resultados
results = {
    'ONNX': [],
    'RKNN': []
}

print("🔧 Configuración del experimento:")
print(f"   • Iteraciones de Monte Carlo: {MONTE_CARLO_ITERATIONS}")
print(f"   • Nivel de confianza: {CONFIDENCE_LEVEL}")
print(f"   • Intervalo de monitoreo: {MONITORING_INTERVAL}s")
print(f"   • Imágenes por experimento: {SAMPLE_SIZE_PER_EXP}")
print(f"   • Carpetas de test disponibles: {len(TEST_FOLDERS)}")

## 3. Funciones de Monitoreo de CPU

In [ ]:
class CPUMonitor:
    """Clase para monitorear el uso de CPU durante la inferencia"""
    
    def __init__(self, interval=0.1):
        self.interval = interval
        self.cpu_data = []
        self.stop_monitoring = False
        self.monitoring_thread = None
        self.num_cpus = psutil.cpu_count(logical=True)
        
    def _monitor_loop(self):
        """Loop de monitoreo de CPU en hilo separado"""
        while not self.stop_monitoring:
            try:
                # Obtener uso por núcleo
                cpu_usage = psutil.cpu_percent(interval=self.interval, percpu=True)
                timestamp = time.time()
                
                self.cpu_data.append({
                    'timestamp': timestamp,
                    'usage_per_core': cpu_usage,
                    'total_usage': np.mean(cpu_usage),
                    'max_usage': np.max(cpu_usage),
                    'min_usage': np.min(cpu_usage),
                    'std_usage': np.std(cpu_usage)
                })
            except Exception as e:
                print(f"Error en monitoreo: {e}")
                
    def start_monitoring(self):
        """Iniciar monitoreo de CPU"""
        self.cpu_data = []
        self.stop_monitoring = False
        self.monitoring_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.monitoring_thread.start()
        
    def stop_monitoring_and_get_data(self):
        """Detener monitoreo y obtener datos"""
        self.stop_monitoring = True
        if self.monitoring_thread:
            self.monitoring_thread.join(timeout=1.0)
        return self.cpu_data.copy()
    
    def get_statistics(self):
        """Calcular estadísticas del uso de CPU"""
        if not self.cpu_data:
            return None
            
        # Extraer datos por núcleo
        usage_matrix = np.array([data['usage_per_core'] for data in self.cpu_data])
        total_usage = np.array([data['total_usage'] for data in self.cpu_data])
        
        stats = {
            'mean_total': np.mean(total_usage),
            'std_total': np.std(total_usage),
            'max_total': np.max(total_usage),
            'min_total': np.min(total_usage),
            'median_total': np.median(total_usage),
            'percentile_95': np.percentile(total_usage, 95),
            'percentile_99': np.percentile(total_usage, 99),
            'mean_per_core': np.mean(usage_matrix, axis=0),
            'std_per_core': np.std(usage_matrix, axis=0),
            'max_per_core': np.max(usage_matrix, axis=0),
            'usage_matrix': usage_matrix,
            'total_usage_series': total_usage,
            'num_samples': len(self.cpu_data)
        }
        
        return stats

# Función auxiliar para calcular intervalos de confianza
def calculate_confidence_interval(data, confidence=0.95):
    """Calcular intervalo de confianza para una muestra"""
    n = len(data)
    mean = np.mean(data)
    std_err = stats.sem(data)
    h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
    return mean - h, mean + h

print("🔧 Funciones de monitoreo de CPU configuradas")
print(f"   • Núcleos de CPU a monitorear: {psutil.cpu_count(logical=True)}")
print(f"   • Intervalo de monitoreo: {MONITORING_INTERVAL}s")

## 4. Pipeline de Procesamiento de Imágenes

In [ ]:
class ImageProcessingPipeline:
    """Pipeline unificado para procesamiento con modelos ONNX y RKNN"""
    
    def __init__(self, model_type="ONNX"):
        self.model_type = model_type
        self.config = MODEL_CONFIGS[model_type]
        self.convert = TextConverter()
        self.models = {}
        
        # Normalización ImageNet para ONNX
        self.MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)
        
    def load_models(self):
        """Cargar modelos según el tipo especificado"""
        try:
            if self.model_type == "ONNX":
                self.models['yolo'] = YOLO(self.config['yolo_path'], task="obb")
                self.models['ocr'] = ort.InferenceSession(self.config['ocr_path'], 
                                                        providers=['CPUExecutionProvider'])
                self.models['segrot'] = ort.InferenceSession(self.config['segrot_path'], 
                                                           providers=['CPUExecutionProvider'])
            elif self.model_type == "RKNN":
                self.models['yolo'] = YOLO(self.config['yolo_path'], task="obb")
                
                # Modelos RKNN
                self.models['ocr'] = RKNNLite()
                self.models['segrot'] = RKNNLite()
                
                self.models['ocr'].load_rknn(self.config['ocr_path'])
                self.models['segrot'].load_rknn(self.config['segrot_path'])
                
                self.models['ocr'].init_runtime()
                self.models['segrot'].init_runtime()
                
            print(f"✅ Modelos {self.model_type} cargados exitosamente")
            return True
            
        except Exception as e:
            print(f"❌ Error cargando modelos {self.model_type}: {e}")
            return False
    
    def preprocess_image(self, image, size):
        """Preprocesamiento de imagen según el tipo de modelo"""
        image_resized = cv2.resize(image, size)
        
        if self.config['normalization']:  # ONNX
            image_processed = np.transpose(image_resized, (2, 0, 1))  # HWC to CHW
            image_processed = np.expand_dims(image_processed, axis=0)  # Add batch dimension
            image_processed = image_processed.astype(np.float32) / 255.0
            image_processed = (image_processed - self.MEAN) / self.STD
        else:  # RKNN
            image_processed = np.expand_dims(image_resized, axis=0)  # Add batch dimension
            
        return image_processed
    
    def rotate_image(self, image, angle):
        """Rotar imagen por un ángulo específico"""
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(image, M, (w, h))
        return rotated
    
    def seg_rot(self, image):
        """Segmentación y rotación de imagen"""
        w, h = image.shape[1], image.shape[0]
        img_trans = self.preprocess_image(image, (224, 224))
        
        # Inferencia según tipo de modelo
        if self.model_type == "ONNX":
            results = self.models['segrot'].run(None, {self.models['segrot'].get_inputs()[0].name: img_trans})
        else:  # RKNN
            results = self.models['segrot'].inference(inputs=[img_trans])
            
        mask, angle = results
        angle = np.squeeze(angle)
        mask = np.squeeze(mask)
        mask = (mask > 0.5).astype(np.uint8) * 255
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        angle = np.argmax(angle) * 5
        
        # Rotar imagen y máscara
        img = self.rotate_image(image, -angle)
        predicted_mask = self.rotate_image(mask, -angle)
        
        # Extraer contorno principal
        contours, _ = cv2.findContours(predicted_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            cnt = max(contours, key=cv2.contourArea)
            rect = cv2.minAreaRect(cnt)
            box = cv2.boxPoints(rect)
            box = np.intp(box)
            
            # Puntos fuente
            src_pts = box.astype("float32")
            
            # Ordenar puntos consistentemente
            rect_pts = np.zeros((4, 2), dtype="float32")
            s = src_pts.sum(axis=1)
            rect_pts[0] = src_pts[np.argmin(s)]  # top-left
            rect_pts[2] = src_pts[np.argmax(s)]  # bottom-right
            
            diff = np.diff(src_pts, axis=1)
            rect_pts[1] = src_pts[np.argmin(diff)]  # top-right
            rect_pts[3] = src_pts[np.argmax(diff)]  # bottom-left
            
            # Calcular dimensiones
            W = int(np.linalg.norm(rect_pts[0] - rect_pts[1]))
            H = int(np.linalg.norm(rect_pts[1] - rect_pts[2]))
            
            if W > 0 and H > 0:
                dst_pts = np.array([[0, 0], [W-1, 0], [W-1, H-1], [0, H-1]], dtype="float32")
                M = cv2.getPerspectiveTransform(rect_pts, dst_pts)
                warped = cv2.warpPerspective(img, M, (W, H))
                return warped
        
        return None
    
    def order_points(self, pts):
        """Ordenar puntos del rectángulo"""
        rect = np.zeros((4, 2), dtype="float32")
        s = pts.sum(axis=1)
        rect[0] = pts[np.argmin(s)]  # top-left
        rect[2] = pts[np.argmax(s)]  # bottom-right
        diff = np.diff(pts, axis=1)
        rect[1] = pts[np.argmin(diff)]  # top-right
        rect[3] = pts[np.argmax(diff)]  # bottom-left
        return rect
    
    def detect_fields(self, image):
        """Detección de campos en imagen"""
        etiquetas = {0: "apellido", 1: "nombre", 2: "numero_cedula"}
        result = self.models['yolo'](image, verbose=False, conf=0.75)
        boxes = result[0].obb
        recortes = []
        
        for j, box in enumerate(boxes):
            cls_id = int(box.cls[0])
            if cls_id in etiquetas:
                pts = box.xyxyxyxy[0].cpu().numpy().reshape(4, 2)
                rect = self.order_points(pts)
                
                # Calcular dimensiones
                (tl, tr, br, bl) = rect
                widthA = np.linalg.norm(br - bl)
                widthB = np.linalg.norm(tr - tl)
                heightA = np.linalg.norm(tr - br)
                heightB = np.linalg.norm(tl - bl)
                
                maxWidth = int(max(widthA, widthB))
                maxHeight = int(max(heightA, heightB))
                
                # Forzar lado largo horizontal
                if maxHeight > maxWidth:
                    maxWidth, maxHeight = maxHeight, maxWidth
                    dst = np.array([[0, 0], [0, maxHeight - 1], [maxWidth - 1, maxHeight - 1], [maxWidth - 1, 0]], dtype="float32")
                else:
                    dst = np.array([[0, 0], [maxWidth - 1, 0], [maxWidth - 1, maxHeight - 1], [0, maxHeight - 1]], dtype="float32")
                
                # Transformación de perspectiva
                M = cv2.getPerspectiveTransform(rect, dst)
                warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
                
                recortes.append({
                    "array": warped,
                    "tipo": etiquetas[cls_id]
                })
        
        return recortes
    
    def run_ocr(self, image_crop):
        """Ejecutar OCR en un recorte de imagen"""
        img_ocr_proc = self.preprocess_image(image_crop, (224, 112))
        
        if self.model_type == "ONNX":
            results_ocr = self.models['ocr'].run(None, {self.models['ocr'].get_inputs()[0].name: img_ocr_proc})
        else:  # RKNN
            results_ocr = self.models['ocr'].inference(inputs=[img_ocr_proc])
            
        results = results_ocr[0]
        text, confidence = self.convert.decode(results, softmax_probs=True)
        return text, confidence
    
    def process_image(self, image_path):
        """Procesar una imagen completa a través del pipeline"""
        try:
            # Cargar imagen
            image = cv2.imread(image_path)
            if image is None:
                return None
                
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Pipeline de procesamiento
            img_rec = self.seg_rot(image)
            if img_rec is None:
                return None
            
            # Detección de campos
            recortes = self.detect_fields(img_rec)
            if not recortes:
                return None
            
            # OCR en cada recorte
            ocr_results = []
            for rec in recortes:
                text, confidence = self.run_ocr(rec['array'])
                ocr_results.append({
                    'field': rec['tipo'],
                    'text': text,
                    'confidence': confidence
                })
            
            return {
                'image_path': image_path,
                'fields_detected': len(recortes),
                'ocr_results': ocr_results
            }
            
        except Exception as e:
            print(f"Error procesando {image_path}: {e}")
            return None

print("🔧 Pipeline de procesamiento configurado para modelos ONNX y RKNN")

## 5. Ejecutar Experimentos de Monte Carlo

In [ ]:
def get_random_images(folders, sample_size):
    """Obtener muestra aleatoria de imágenes de las carpetas especificadas"""
    all_images = []
    
    for folder in folders:
        if os.path.exists(folder):
            # Buscar archivos de imagen
            extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
            for ext in extensions:
                all_images.extend(glob(os.path.join(folder, ext)))
    
    if len(all_images) < sample_size:
        print(f"⚠️  Solo se encontraron {len(all_images)} imágenes, usando todas")
        return all_images
    
    # Muestra aleatoria
    return random.sample(all_images, sample_size)

def run_monte_carlo_experiment():
    """Ejecutar el experimento completo de Monte Carlo"""
    
    print("🚀 Iniciando experimento de Monte Carlo")
    print("=" * 60)
    
    for model_type in ['ONNX', 'RKNN']:
        print(f"\n📊 Ejecutando experimentos para modelos {model_type}")
        print("-" * 40)
        
        # Inicializar pipeline
        pipeline = ImageProcessingPipeline(model_type)
        
        if not pipeline.load_models():
            print(f"❌ No se pudieron cargar los modelos {model_type}, saltando...")
            continue
        
        # Ejecutar múltiples experimentos
        for exp_num in range(1, MONTE_CARLO_ITERATIONS + 1):
            print(f"\n🔬 Experimento {exp_num}/{MONTE_CARLO_ITERATIONS} - {model_type}")
            
            # Obtener muestra aleatoria de imágenes
            sample_images = get_random_images(TEST_FOLDERS, SAMPLE_SIZE_PER_EXP)
            
            if not sample_images:
                print("❌ No se encontraron imágenes para procesar")
                continue
            
            # Detectar de qué folders vienen las imágenes
            folders_used = set()
            for img_path in sample_images:
                for test_folder in TEST_FOLDERS:
                    if test_folder in img_path:
                        folders_used.add(test_folder)
                        break
            
            print(f"📁 Procesando {len(sample_images)} imágenes:")
            for img in sample_images:
                print(f"   • {os.path.basename(img)}")
            
            # Configurar monitor de CPU
            cpu_monitor = CPUMonitor(interval=MONITORING_INTERVAL)
            
            # Iniciar monitoreo y procesamiento
            start_time = time.time()
            cpu_monitor.start_monitoring()
            
            # Procesar cada imagen
            processing_results = []
            for img_path in sample_images:
                result = pipeline.process_image(img_path)
                if result:
                    processing_results.append(result)
                    print(f"✅ {os.path.basename(img_path)}: {result['fields_detected']} campos detectados")
                else:
                    print(f"⚠️  {os.path.basename(img_path)}: Error en procesamiento")
            
            # Detener monitoreo
            end_time = time.time()
            cpu_data = cpu_monitor.stop_monitoring_and_get_data()
            cpu_stats = cpu_monitor.get_statistics()
            
            # Almacenar resultados del experimento
            experiment_result = {
                'experiment_number': exp_num,
                'model_type': model_type,
                'start_time': start_time,
                'end_time': end_time,
                'duration': end_time - start_time,
                'images_processed': len(processing_results),
                'images_failed': len(sample_images) - len(processing_results),
                'sample_images': [os.path.basename(img) for img in sample_images],
                'sample_images_full_path': sample_images,
                'folders_used': list(folders_used),
                'cpu_stats': cpu_stats,
                'cpu_raw_data': cpu_data,
                'processing_results': processing_results
            }
            
            results[model_type].append(experiment_result)
            
            # Mostrar resumen del experimento
            if cpu_stats:
                print(f"⏱️  Duración: {experiment_result['duration']:.2f}s")
                print(f"📈 CPU promedio: {cpu_stats['mean_total']:.1f}%")
                print(f"📊 CPU máximo: {cpu_stats['max_total']:.1f}%")
                print(f"📉 CPU mínimo: {cpu_stats['min_total']:.1f}%")
                print(f"📋 Muestras CPU: {cpu_stats['num_samples']}")
            
            # Pausa entre experimentos para estabilización
            time.sleep(2)
    
    print("\n🎉 Experimentos de Monte Carlo completados")
    print("=" * 60)
    
    # Resumen general
    for model_type in results:
        if results[model_type]:
            print(f"\n📊 Resumen {model_type}:")
            print(f"   • Experimentos completados: {len(results[model_type])}")
            
            # Estadísticas agregadas
            durations = [exp['duration'] for exp in results[model_type]]
            cpu_means = [exp['cpu_stats']['mean_total'] for exp in results[model_type] if exp['cpu_stats']]
            
            if durations:
                print(f"   • Duración promedio: {np.mean(durations):.2f}s ± {np.std(durations):.2f}s")
            if cpu_means:
                print(f"   • CPU promedio: {np.mean(cpu_means):.1f}% ± {np.std(cpu_means):.1f}%")

# Ejecutar el experimento
run_monte_carlo_experiment()

In [ ]:
# Test básico de funcionamiento antes del experimento completo
print("🧪 PRUEBA RÁPIDA DE FUNCIONAMIENTO")
print("=" * 50)

# Verificar que tenemos imágenes disponibles
test_images = get_random_images(TEST_FOLDERS, 2)
print(f"✅ Imágenes encontradas: {len(test_images)}")
for img in test_images[:3]:
    print(f"   📁 {os.path.basename(img)}")

# Test rápido de CPU monitor
print("\n🔧 Test de monitor de CPU...")
cpu_monitor = CPUMonitor(interval=0.1)
cpu_monitor.start_monitoring()
time.sleep(1)  # Monitor por 1 segundo
cpu_data = cpu_monitor.stop_monitoring_and_get_data()
cpu_stats = cpu_monitor.get_statistics()

if cpu_stats:
    print(f"   ✅ Monitor funcionando: {cpu_stats['num_samples']} muestras recolectadas")
    print(f"   📊 CPU promedio durante test: {cpu_stats['mean_total']:.1f}%")
else:
    print("   ❌ Error en monitor de CPU")

print("\n🎯 Sistema listo para experimento completo!")

## 6. Análisis Estadístico del Uso de CPU

In [ ]:
def analyze_experiment_statistics():
    """Realizar análisis estadístico detallado de los experimentos"""
    
    print("📊 ANÁLISIS ESTADÍSTICO DE EXPERIMENTOS")
    print("=" * 50)
    
    analysis_results = {}
    
    for model_type in ['ONNX', 'RKNN']:
        if not results[model_type]:
            print(f"⚠️  No hay datos para {model_type}")
            continue
            
        print(f"\n🔍 Análisis para modelos {model_type}:")
        print("-" * 30)
        
        # Extraer métricas de todos los experimentos
        durations = []
        cpu_means = []
        cpu_stds = []
        cpu_maxes = []
        cpu_mins = []
        total_usage_series = []
        
        for exp in results[model_type]:
            if exp['cpu_stats']:
                durations.append(exp['duration'])
                cpu_means.append(exp['cpu_stats']['mean_total'])
                cpu_stds.append(exp['cpu_stats']['std_total'])
                cpu_maxes.append(exp['cpu_stats']['max_total'])
                cpu_mins.append(exp['cpu_stats']['min_total'])
                total_usage_series.extend(exp['cpu_stats']['total_usage_series'])
        
        if not cpu_means:
            print(f"❌ No hay datos válidos de CPU para {model_type}")
            continue
            
        # Convertir a arrays numpy
        durations = np.array(durations)
        cpu_means = np.array(cpu_means)
        cpu_stds = np.array(cpu_stds)
        cpu_maxes = np.array(cpu_maxes)
        cpu_mins = np.array(cpu_mins)
        total_usage_series = np.array(total_usage_series)
        
        # Calcular estadísticas descriptivas
        stats = {
            # Tiempos de procesamiento
            'duration_mean': np.mean(durations),
            'duration_std': np.std(durations),
            'duration_min': np.min(durations),
            'duration_max': np.max(durations),
            'duration_median': np.median(durations),
            
            # Uso promedio de CPU por experimento
            'cpu_mean_avg': np.mean(cpu_means),
            'cpu_mean_std': np.std(cpu_means),
            'cpu_mean_min': np.min(cpu_means),
            'cpu_mean_max': np.max(cpu_means),
            'cpu_mean_median': np.median(cpu_means),
            
            # Variabilidad del CPU por experimento
            'cpu_std_avg': np.mean(cpu_stds),
            'cpu_std_std': np.std(cpu_stds),
            
            # Picos de CPU
            'cpu_max_avg': np.mean(cpu_maxes),
            'cpu_max_std': np.std(cpu_maxes),
            'cpu_max_max': np.max(cpu_maxes),
            
            # Mínimos de CPU
            'cpu_min_avg': np.mean(cpu_mins),
            'cpu_min_std': np.std(cpu_mins),
            'cpu_min_min': np.min(cpu_mins),
            
            # Estadísticas globales de CPU
            'global_cpu_mean': np.mean(total_usage_series),
            'global_cpu_std': np.std(total_usage_series),
            'global_cpu_median': np.median(total_usage_series),
            'global_cpu_p95': np.percentile(total_usage_series, 95),
            'global_cpu_p99': np.percentile(total_usage_series, 99),
            
            # Intervalos de confianza
            'duration_ci': calculate_confidence_interval(durations, CONFIDENCE_LEVEL),
            'cpu_mean_ci': calculate_confidence_interval(cpu_means, CONFIDENCE_LEVEL),
            'cpu_max_ci': calculate_confidence_interval(cpu_maxes, CONFIDENCE_LEVEL),
            
            # Series temporales para visualización
            'durations_series': durations,
            'cpu_means_series': cpu_means,
            'cpu_maxes_series': cpu_maxes,
            'total_usage_series': total_usage_series,
            
            # Metadatos
            'num_experiments': len(durations),
            'total_samples': len(total_usage_series)
        }
        
        analysis_results[model_type] = stats
        
        # Mostrar resultados
        print(f"📈 Tiempo de procesamiento:")
        print(f"   • Promedio: {stats['duration_mean']:.2f}s ± {stats['duration_std']:.2f}s")
        print(f"   • Rango: [{stats['duration_min']:.2f}s - {stats['duration_max']:.2f}s]")
        print(f"   • IC 95%: [{stats['duration_ci'][0]:.2f}s - {stats['duration_ci'][1]:.2f}s]")
        
        print(f"\n💻 Uso de CPU (promedio por experimento):")
        print(f"   • Promedio: {stats['cpu_mean_avg']:.1f}% ± {stats['cpu_mean_std']:.1f}%")
        print(f"   • Rango: [{stats['cpu_mean_min']:.1f}% - {stats['cpu_mean_max']:.1f}%]")
        print(f"   • IC 95%: [{stats['cpu_mean_ci'][0]:.1f}% - {stats['cpu_mean_ci'][1]:.1f}%]")
        
        print(f"\n🔥 Picos de CPU:")
        print(f"   • Promedio: {stats['cpu_max_avg']:.1f}% ± {stats['cpu_max_std']:.1f}%")
        print(f"   • Máximo absoluto: {stats['cpu_max_max']:.1f}%")
        print(f"   • IC 95%: [{stats['cpu_max_ci'][0]:.1f}% - {stats['cpu_max_ci'][1]:.1f}%]")
        
        print(f"\n📊 Estadísticas globales CPU:")
        print(f"   • Media global: {stats['global_cpu_mean']:.1f}%")
        print(f"   • Desv. estándar: {stats['global_cpu_std']:.1f}%")
        print(f"   • Percentil 95: {stats['global_cpu_p95']:.1f}%")
        print(f"   • Percentil 99: {stats['global_cpu_p99']:.1f}%")
        
        print(f"\n📋 Metadatos:")
        print(f"   • Experimentos: {stats['num_experiments']}")
        print(f"   • Muestras totales CPU: {stats['total_samples']}")
    
    return analysis_results

# Ejecutar análisis estadístico
statistical_analysis = analyze_experiment_statistics()

## 7. Visualización Comparativa de Distribuciones

In [ ]:
def create_comprehensive_visualizations():
    """Crear visualizaciones completas de los resultados y guardar imágenes individuales"""
    
    if not any(results.values()):
        print("❌ No hay datos para visualizar")
        return
    
    # Crear directorio para guardar imágenes
    output_dir = "monte_carlo_visualizations"
    os.makedirs(output_dir, exist_ok=True)
    
    print("🎨 Generando visualizaciones comprehensivas...")
    print(f"📁 Guardando imágenes en: {output_dir}/")
    
    # Función auxiliar para guardar figuras individuales
    def save_individual_figure(fig, filename_base):
        """Guardar figura en SVG y PNG"""
        svg_path = os.path.join(output_dir, f"{filename_base}.svg")
        png_path = os.path.join(output_dir, f"{filename_base}.png")
        
        fig.savefig(svg_path, format='svg', bbox_inches='tight', dpi=300)
        fig.savefig(png_path, format='png', bbox_inches='tight', dpi=300)
        
        print(f"💾 Guardado: {filename_base}.svg y {filename_base}.png")
    
    # ==========================================
    # 1. DISTRIBUCIÓN DE TIEMPOS DE PROCESAMIENTO
    # ==========================================
    print("\n📊 1. Generando distribución de tiempos de procesamiento...")
    fig_time_dist = plt.figure(figsize=(12, 6))
    
    for i, model_type in enumerate(['ONNX', 'RKNN']):
        if model_type not in statistical_analysis:
            continue
            
        ax = fig_time_dist.add_subplot(1, 2, i+1)
        
        # Obtener datos de tiempo
        durations = statistical_analysis[model_type]['durations_series']
        
        # Histograma
        n, bins, patches = ax.hist(durations, bins=15, alpha=0.7, 
                                  color=MODEL_CONFIGS[model_type]['color'],
                                  density=True, edgecolor='black', linewidth=1.2,
                                  label=f'Datos {model_type}')
        
        # Curva de densidad KDE
        x = np.linspace(durations.min(), durations.max(), 100)
        kde = stats.gaussian_kde(durations)
        ax.plot(x, kde(x), 'k-', linewidth=2.5, label='Densidad KDE')
        
        # Estadísticas en el gráfico
        mean_val = statistical_analysis[model_type]['duration_mean']
        std_val = statistical_analysis[model_type]['duration_std']
        median_val = statistical_analysis[model_type]['duration_median']
        
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.2f}s')
        ax.axvline(median_val, color='blue', linestyle='-.', linewidth=2, label=f'Mediana: {median_val:.2f}s')
        ax.axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=1.5, label=f'+1σ: {mean_val + std_val:.2f}s')
        ax.axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=1.5, label=f'-1σ: {mean_val - std_val:.2f}s')
        
        ax.set_title(f'Distribución de Tiempos - {model_type}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Tiempo de Procesamiento (s)', fontsize=12)
        ax.set_ylabel('Densidad', fontsize=12)
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Distribución del Tiempo de Procesamiento', fontsize=16, fontweight='bold')
    plt.tight_layout()
    save_individual_figure(fig_time_dist, "01_distribucion_tiempos")
    plt.show()
    
    # ==========================================
    # 2. HISTOGRAMAS DE DISTRIBUCIÓN CPU
    # ==========================================
    print("📊 2. Generando histogramas de distribución CPU...")
    fig_hist = plt.figure(figsize=(10, 12))
    
    for i, model_type in enumerate(['ONNX', 'RKNN']):
        if model_type not in statistical_analysis:
            continue
            
        ax = fig_hist.add_subplot(2, 1, i+1)
        data = statistical_analysis[model_type]['total_usage_series']
        
        # Histograma
        n, bins, patches = ax.hist(data, bins=30, alpha=0.7, 
                                  color=MODEL_CONFIGS[model_type]['color'],
                                  density=True, label=f'Datos {model_type}')
        
        # Curva de densidad
        x = np.linspace(data.min(), data.max(), 100)
        kde = stats.gaussian_kde(data)
        ax.plot(x, kde(x), 'k-', linewidth=2, label='Densidad KDE')
        
        # Estadísticas en el gráfico
        mean_val = statistical_analysis[model_type]['global_cpu_mean']
        std_val = statistical_analysis[model_type]['global_cpu_std']
        
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.1f}%')
        ax.axvline(mean_val + std_val, color='orange', linestyle=':', label=f'+1σ: {mean_val + std_val:.1f}%')
        ax.axvline(mean_val - std_val, color='orange', linestyle=':', label=f'-1σ: {mean_val - std_val:.1f}%')
        
        ax.set_title(f'Distribución CPU - {model_type}', fontweight='bold')
        ax.set_xlabel('Uso CPU (%)')
        ax.set_ylabel('Densidad')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Distribuciones de Uso de CPU', fontsize=16, fontweight='bold')
    plt.tight_layout()
    save_individual_figure(fig_hist, "02_histogramas_distribucion_cpu")
    plt.show()
    
    # ==========================================
    # 3. GRÁFICO DE VIOLÍN COMPARATIVO
    # ==========================================
    print("📊 3. Generando gráfico de violín...")
    fig_violin = plt.figure(figsize=(10, 6))
    ax = fig_violin.add_subplot(1, 1, 1)
    
    # Preparar datos para violín
    violin_data = []
    positions = []
    pos = 1
    
    for model_type in ['ONNX', 'RKNN']:
        if model_type in statistical_analysis:
            data = statistical_analysis[model_type]['total_usage_series']
            violin_data.append(data)
            positions.append(pos)
            pos += 1
    
    if violin_data:
        parts = ax.violinplot(violin_data, positions=positions, widths=0.7, showmeans=True)
        
        # Colorear violines
        for i, (pc, model_type) in enumerate(zip(parts['bodies'], ['ONNX', 'RKNN'])):
            if model_type in MODEL_CONFIGS:
                pc.set_facecolor(MODEL_CONFIGS[model_type]['color'])
                pc.set_alpha(0.7)
    
    ax.set_title('Distribución de Densidad CPU (Violín)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tipo de Modelo')
    ax.set_ylabel('Uso CPU (%)')
    ax.set_xticks(positions)
    ax.set_xticklabels([m for m in ['ONNX', 'RKNN'] if m in statistical_analysis])
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_individual_figure(fig_violin, "03_violin_plot_cpu")
    plt.show()
    
    # ==========================================
    # 4. MÉTRICAS DE RENDIMIENTO
    # ==========================================
    print("📊 4. Generando métricas de rendimiento...")
    fig_perf = plt.figure(figsize=(10, 12))
    
    # Tiempo de procesamiento
    if len([m for m in ['ONNX', 'RKNN'] if m in statistical_analysis]) >= 2:
        models = []
        durations = []
        duration_errors = []
        
        for model_type in ['ONNX', 'RKNN']:
            if model_type in statistical_analysis:
                models.append(model_type)
                durations.append(statistical_analysis[model_type]['duration_mean'])
                duration_errors.append(statistical_analysis[model_type]['duration_std'])
        
        ax1 = fig_perf.add_subplot(2, 1, 1)
        bars1 = ax1.bar(models, durations, yerr=duration_errors, 
                       color=[MODEL_CONFIGS[m]['color'] for m in models],
                       alpha=0.7, capsize=5)
        ax1.set_title('Tiempo de Procesamiento Promedio', fontweight='bold')
        ax1.set_ylabel('Tiempo (s)')
        ax1.grid(True, alpha=0.3)
        
        # Agregar valores en las barras
        for bar, val, err in zip(bars1, durations, duration_errors):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + err,
                    f'{val:.2f}s\n±{err:.2f}s',
                    ha='center', va='bottom', fontsize=10)
        
        # CPU promedio
        cpu_means = []
        cpu_errors = []
        
        for model_type in models:
            cpu_means.append(statistical_analysis[model_type]['cpu_mean_avg'])
            cpu_errors.append(statistical_analysis[model_type]['cpu_mean_std'])
        
        ax2 = fig_perf.add_subplot(2, 1, 2)
        bars2 = ax2.bar(models, cpu_means, yerr=cpu_errors,
                       color=[MODEL_CONFIGS[m]['color'] for m in models],
                       alpha=0.7, capsize=5)
        ax2.set_title('Uso Promedio de CPU', fontweight='bold')
        ax2.set_ylabel('CPU (%)')
        ax2.grid(True, alpha=0.3)
        
        # Agregar valores en las barras
        for bar, val, err in zip(bars2, cpu_means, cpu_errors):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + err,
                    f'{val:.1f}%\n±{err:.1f}%',
                    ha='center', va='bottom', fontsize=10)
    
    plt.suptitle('Métricas de Rendimiento - Tiempo vs CPU', fontsize=16, fontweight='bold')
    plt.tight_layout()
    save_individual_figure(fig_perf, "04_metricas_rendimiento")
    plt.show()
    
    # ==========================================
    # 5. HEATMAP DE MEDIA CPU: FOLDERS VS CPUS
    # ==========================================
    print("📊 5. Generando heatmap de CPU por folder y núcleo...")
    
    if psutil.cpu_count(logical=True) <= 16:  # Solo si no hay demasiados núcleos
        fig_heat = plt.figure(figsize=(14, 10))
        
        for i, model_type in enumerate(['ONNX', 'RKNN']):
            if model_type not in results or not results[model_type]:
                continue
                
            ax = fig_heat.add_subplot(2, 1, i+1)
            
            # Recopilar datos por folder y núcleo - calculando media de todos los experimentos
            folder_cpu_data = defaultdict(lambda: defaultdict(list))
            
            for exp in results[model_type]:
                if exp['cpu_stats'] and 'usage_matrix' in exp['cpu_stats']:
                    # Un experimento puede tener imágenes de múltiples folders
                    folders_in_exp = exp.get('folders_used', [])
                    usage_matrix = exp['cpu_stats']['usage_matrix']
                    
                    # Para cada folder usado en este experimento, agregar datos
                    for folder_name in folders_in_exp:
                        # Para cada núcleo, calcular la media de ese experimento
                        for core_idx in range(usage_matrix.shape[1]):
                            core_mean = np.mean(usage_matrix[:, core_idx])
                            folder_cpu_data[folder_name][core_idx].append(core_mean)
            
            if folder_cpu_data:
                # Ordenar folders
                folders_sorted = sorted(folder_cpu_data.keys())
                num_cores = psutil.cpu_count(logical=True)
                
                # Crear matriz: folders (filas) x CPUs (columnas)
                # Cada celda es la media de todos los experimentos para ese folder y CPU
                heatmap_matrix = np.zeros((len(folders_sorted), num_cores))
                
                for folder_idx, folder_name in enumerate(folders_sorted):
                    for core_idx in range(num_cores):
                        if core_idx in folder_cpu_data[folder_name]:
                            # Media de todos los experimentos
                            heatmap_matrix[folder_idx, core_idx] = np.mean(folder_cpu_data[folder_name][core_idx])
                
                # Crear heatmap
                im = ax.imshow(heatmap_matrix, cmap='YlOrRd', aspect='auto', 
                              vmin=0, vmax=100, interpolation='nearest')
                
                # Configurar etiquetas
                ax.set_xticks(range(num_cores))
                ax.set_xticklabels([f'CPU{i}' for i in range(num_cores)], fontsize=9)
                ax.set_yticks(range(len(folders_sorted)))
                ax.set_yticklabels([f.split('/')[-1] for f in folders_sorted], fontsize=10)
                
                # Agregar valores en las celdas
                for y in range(len(folders_sorted)):
                    for x in range(num_cores):
                        value = heatmap_matrix[y, x]
                        text_color = "white" if value > 50 else "black"
                        ax.text(x, y, f'{value:.1f}',
                               ha="center", va="center", color=text_color, fontsize=8, fontweight='bold')
                
                ax.set_title(f'Heatmap CPU por Folder - {model_type}\n(Media de todos los experimentos)', 
                           fontsize=12, fontweight='bold')
                ax.set_xlabel('Núcleo de CPU', fontsize=11)
                ax.set_ylabel('Folder de Test', fontsize=11)
                
                # Agregar colorbar
                cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                cbar.set_label('Uso CPU Promedio (%)', fontsize=10)
        
        plt.suptitle('Heatmap: Folders vs CPUs (Media de Experimentos)', fontsize=16, fontweight='bold')
        plt.tight_layout()
        save_individual_figure(fig_heat, "05_heatmap_folders_vs_cpus")
        plt.show()
    else:
        print("⚠️  Demasiados núcleos para visualizar heatmap")
    
    # ==========================================
    # 6. FIGURA COMPLETA RESUMIDA
    # ==========================================
    print("📊 6. Generando resumen completo...")
    fig_complete = plt.figure(figsize=(20, 12))
    gs = fig_complete.add_gridspec(3, 4, hspace=0.3, wspace=0.3)
    
    # Distribución de tiempos resumida
    for i, model_type in enumerate(['ONNX', 'RKNN']):
        if model_type not in statistical_analysis:
            continue
        ax = fig_complete.add_subplot(gs[0, i*2:(i+1)*2])
        durations = statistical_analysis[model_type]['durations_series']
        ax.hist(durations, bins=12, alpha=0.7, color=MODEL_CONFIGS[model_type]['color'], 
               density=True, edgecolor='black')
        mean_val = statistical_analysis[model_type]['duration_mean']
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2)
        ax.set_title(f'Distribución Tiempo - {model_type}', fontsize=12)
        ax.set_xlabel('Tiempo (s)', fontsize=10)
        ax.set_ylabel('Densidad', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Histogramas de CPU resumidos
    for i, model_type in enumerate(['ONNX', 'RKNN']):
        if model_type not in statistical_analysis:
            continue
        ax = fig_complete.add_subplot(gs[1, i*2:(i+1)*2])
        data = statistical_analysis[model_type]['total_usage_series']
        ax.hist(data, bins=20, alpha=0.7, color=MODEL_CONFIGS[model_type]['color'], density=True)
        mean_val = statistical_analysis[model_type]['global_cpu_mean']
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5)
        ax.set_title(f'Distribución CPU - {model_type}', fontsize=12)
        ax.set_xlabel('CPU (%)', fontsize=10)
        ax.set_ylabel('Densidad', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Métricas de rendimiento resumidas
    if len([m for m in ['ONNX', 'RKNN'] if m in statistical_analysis]) >= 2:
        models = [m for m in ['ONNX', 'RKNN'] if m in statistical_analysis]
        
        # Tiempo
        ax = fig_complete.add_subplot(gs[2, :2])
        durations = [statistical_analysis[m]['duration_mean'] for m in models]
        bars = ax.bar(models, durations, color=[MODEL_CONFIGS[m]['color'] for m in models], alpha=0.7)
        ax.set_title('Tiempo Procesamiento', fontsize=12)
        ax.set_ylabel('Tiempo (s)', fontsize=10)
        
        # CPU
        ax = fig_complete.add_subplot(gs[2, 2:])
        cpu_means = [statistical_analysis[m]['cpu_mean_avg'] for m in models]
        bars = ax.bar(models, cpu_means, color=[MODEL_CONFIGS[m]['color'] for m in models], alpha=0.7)
        ax.set_title('CPU Promedio', fontsize=12)
        ax.set_ylabel('CPU (%)', fontsize=10)
    
    plt.suptitle('Experimento Monte Carlo: Resumen Ejecutivo ONNX vs RKNN', 
                fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    save_individual_figure(fig_complete, "00_resumen_ejecutivo_completo")
    plt.show()
    
    print("\n✅ Visualizaciones generadas exitosamente")
    print(f"📁 Todas las imágenes guardadas en: {output_dir}/")
    print("📄 Formatos disponibles: SVG (vectorial) y PNG (alta resolución)")

create_comprehensive_visualizations()
# Crear visualizaciones

## 8. Resumen de Métricas de Rendimiento

In [ ]:
import pandas as pd
import numpy as np
import time
import psutil
from scipy.stats import ttest_ind, mannwhitneyu, ks_2samp

def generate_final_summary():
    """Generar resumen final y tabla de métricas de rendimiento"""
    
    print("📋 RESUMEN FINAL DEL EXPERIMENTO DE MONTE CARLO")
    print("=" * 60)
    
    if not any(statistical_analysis.values()):
        print("❌ No hay datos de análisis estadístico disponibles")
        return
    
    # Crear tabla de comparación
    summary_data = []
    
    for model_type in ['ONNX', 'RKNN']:
        if model_type in statistical_analysis:
            stats = statistical_analysis[model_type]
            summary_data.append({
                'Modelo': model_type,
                'Experimentos': stats['num_experiments'],
                'Muestras CPU': stats['total_samples'],
                'Tiempo Promedio (s)': f"{stats['duration_mean']:.2f} ± {stats['duration_std']:.2f}",
                'CPU Promedio (%)': f"{stats['cpu_mean_avg']:.1f} ± {stats['cpu_mean_std']:.1f}",
                'CPU Máximo (%)': f"{stats['cpu_max_avg']:.1f} ± {stats['cpu_max_std']:.1f}",
                'CPU Global (%)': f"{stats['global_cpu_mean']:.1f} ± {stats['global_cpu_std']:.1f}",
                'Percentil 95': f"{stats['global_cpu_p95']:.1f}%",
                'Percentil 99': f"{stats['global_cpu_p99']:.1f}%"
            })
    
    if summary_data:
        # Crear DataFrame para mejor visualización
        df_summary = pd.DataFrame(summary_data)
        
        print("\n📊 TABLA COMPARATIVA DE RENDIMIENTO:")
        print("-" * 80)
        print(df_summary.to_string(index=False))
        
        # Análisis de significancia estadística
        if len(summary_data) == 2:
            print("\n\n🔍 ANÁLISIS DE SIGNIFICANCIA ESTADÍSTICA:")
            print("-" * 50)
            
            onnx_cpu = statistical_analysis['ONNX']['total_usage_series']
            rknn_cpu = statistical_analysis['RKNN']['total_usage_series']
            
            # Test t de Student
            t_stat, t_p_value = ttest_ind(onnx_cpu, rknn_cpu)
            
            # Test de Mann-Whitney U (no paramétrico)
            u_stat, u_p_value = mannwhitneyu(onnx_cpu, rknn_cpu, alternative='two-sided')
            
            # Test de Kolmogorov-Smirnov
            ks_stat, ks_p_value = ks_2samp(onnx_cpu, rknn_cpu)
            
            print(f"📈 Test t de Student:")
            print(f"   • Estadístico t: {t_stat:.4f}")
            print(f"   • p-valor: {t_p_value:.6f}")
            print(f"   • Significativo (α=0.05): {'Sí' if t_p_value < 0.05 else 'No'}")
            
            print(f"\n📊 Test Mann-Whitney U:")
            print(f"   • Estadístico U: {u_stat:.0f}")
            print(f"   • p-valor: {u_p_value:.6f}")
            print(f"   • Significativo (α=0.05): {'Sí' if u_p_value < 0.05 else 'No'}")
            
            print(f"\n📋 Test Kolmogorov-Smirnov:")
            print(f"   • Estadístico KS: {ks_stat:.4f}")
            print(f"   • p-valor: {ks_p_value:.6f}")
            print(f"   • Significativo (α=0.05): {'Sí' if ks_p_value < 0.05 else 'No'}")
            
            # Tamaño del efecto (Cohen's d)
            pooled_std = np.sqrt(((len(onnx_cpu) - 1) * np.var(onnx_cpu, ddof=1) + 
                                 (len(rknn_cpu) - 1) * np.var(rknn_cpu, ddof=1)) / 
                                (len(onnx_cpu) + len(rknn_cpu) - 2))
            cohens_d = (np.mean(onnx_cpu) - np.mean(rknn_cpu)) / pooled_std
            
            print(f"\n📏 Tamaño del Efecto (Cohen's d):")
            print(f"   • d = {cohens_d:.4f}")
            
            if abs(cohens_d) < 0.2:
                effect_size = "Pequeño"
            elif abs(cohens_d) < 0.5:
                effect_size = "Mediano"
            elif abs(cohens_d) < 0.8:
                effect_size = "Grande"
            else:
                effect_size = "Muy grande"
            
            print(f"   • Interpretación: Efecto {effect_size.lower()}")
            
            # Recomendación final
            print(f"\n\n🎯 RECOMENDACIONES:")
            print("-" * 30)
            
            onnx_mean = statistical_analysis['ONNX']['cpu_mean_avg']
            rknn_mean = statistical_analysis['RKNN']['cpu_mean_avg']
            
            onnx_time = statistical_analysis['ONNX']['duration_mean']
            rknn_time = statistical_analysis['RKNN']['duration_mean']
            
            if onnx_mean < rknn_mean:
                cpu_winner = "ONNX"
                cpu_diff = ((rknn_mean - onnx_mean) / rknn_mean) * 100
            else:
                cpu_winner = "RKNN"
                cpu_diff = ((onnx_mean - rknn_mean) / onnx_mean) * 100
            
            if onnx_time < rknn_time:
                time_winner = "ONNX"
                time_diff = ((rknn_time - onnx_time) / rknn_time) * 100
            else:
                time_winner = "RKNN"
                time_diff = ((onnx_time - rknn_time) / onnx_time) * 100
            
            print(f"🏆 Mejor rendimiento CPU: {cpu_winner} ({cpu_diff:.1f}% menos uso)")
            print(f"⚡ Mejor tiempo procesamiento: {time_winner} ({time_diff:.1f}% más rápido)")
            
            # Intervalos de confianza
            print(f"\n📐 INTERVALOS DE CONFIANZA (95%):")
            print("-" * 40)
            
            for model_type in ['ONNX', 'RKNN']:
                stats = statistical_analysis[model_type]
                cpu_ci = stats['cpu_mean_ci']
                duration_ci = stats['duration_ci']
                
                print(f"\n{model_type}:")
                print(f"   • CPU promedio: [{cpu_ci[0]:.1f}% - {cpu_ci[1]:.1f}%]")
                print(f"   • Tiempo procesamiento: [{duration_ci[0]:.2f}s - {duration_ci[1]:.2f}s]")
    
    # Información del experimento
    print(f"\n\n📝 INFORMACIÓN DEL EXPERIMENTO:")
    print("-" * 40)
    print(f"• Fecha de ejecución: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"• Iteraciones Monte Carlo: {MONTE_CARLO_ITERATIONS} por modelo")
    print(f"• Nivel de confianza: {CONFIDENCE_LEVEL * 100}%")
    print(f"• Intervalo de monitoreo CPU: {MONITORING_INTERVAL}s")
    print(f"• Imágenes por experimento: {SAMPLE_SIZE_PER_EXP}")
    print(f"• Núcleos CPU monitoreados: {psutil.cpu_count(logical=True)}")
    
    total_experiments = sum(len(results[model]) for model in results if results[model])
    total_processing_time = sum(
        exp['duration'] for model in results 
        for exp in results[model] if exp['cpu_stats']
    )
    
    print(f"• Total experimentos ejecutados: {total_experiments}")
    print(f"• Tiempo total de procesamiento: {total_processing_time:.2f}s")
    
    print("\n" + "=" * 60)
    print("🎉 EXPERIMENTO DE MONTE CARLO COMPLETADO EXITOSAMENTE")
    print("=" * 60)

# Generar resumen final
generate_final_summary()

In [ ]:
from scipy import stats as scipy_stats

def generate_final_summary():
    """Generar resumen final y tabla de métricas de rendimiento"""
    
    print("📋 RESUMEN FINAL DEL EXPERIMENTO DE MONTE CARLO")
    print("=" * 60)
    
    if not any(statistical_analysis.values()):
        print("❌ No hay datos de análisis estadístico disponibles")
        return
    
    # Crear tabla de comparación
    summary_data = []
    
    for model_type in ['ONNX', 'RKNN']:
        if model_type in statistical_analysis:
            stats = statistical_analysis[model_type]
            summary_data.append({
                'Modelo': model_type,
                'Experimentos': stats['num_experiments'],
                'Muestras CPU': stats['total_samples'],
                'Tiempo Promedio (s)': f"{stats['duration_mean']:.2f} ± {stats['duration_std']:.2f}",
                'CPU Promedio (%)': f"{stats['cpu_mean_avg']:.1f} ± {stats['cpu_mean_std']:.1f}",
                'CPU Máximo (%)': f"{stats['cpu_max_avg']:.1f} ± {stats['cpu_max_std']:.1f}",
                'CPU Global (%)': f"{stats['global_cpu_mean']:.1f} ± {stats['global_cpu_std']:.1f}",
                'Percentil 95': f"{stats['global_cpu_p95']:.1f}%",
                'Percentil 99': f"{stats['global_cpu_p99']:.1f}%"
            })
    
    if summary_data:
        # Crear DataFrame para mejor visualización
        df_summary = pd.DataFrame(summary_data)
        
        print("\n📊 TABLA COMPARATIVA DE RENDIMIENTO:")
        print("-" * 80)
        print(df_summary.to_string(index=False))
        
        # Análisis de significancia estadística
        if len(summary_data) == 2:
            print("\n\n🔍 ANÁLISIS DE SIGNIFICANCIA ESTADÍSTICA:")
            print("-" * 50)
            
            onnx_cpu = statistical_analysis['ONNX']['total_usage_series']
            rknn_cpu = statistical_analysis['RKNN']['total_usage_series']
            
            # Test t de Student
            t_stat, t_p_value = scipy_stats.ttest_ind(onnx_cpu, rknn_cpu)
            
            # Test de Mann-Whitney U (no paramétrico)
            u_stat, u_p_value = scipy_stats.mannwhitneyu(onnx_cpu, rknn_cpu, alternative='two-sided')
            
            # Test de Kolmogorov-Smirnov
            ks_stat, ks_p_value = scipy_stats.ks_2samp(onnx_cpu, rknn_cpu)
            
            print(f"📈 Test t de Student:")
            print(f"   • Estadístico t: {t_stat:.4f}")
            print(f"   • p-valor: {t_p_value:.6f}")
            print(f"   • Significativo (α=0.05): {'Sí' if t_p_value < 0.05 else 'No'}")
            
            print(f"\n📊 Test Mann-Whitney U:")
            print(f"   • Estadístico U: {u_stat:.0f}")
            print(f"   • p-valor: {u_p_value:.6f}")
            print(f"   • Significativo (α=0.05): {'Sí' if u_p_value < 0.05 else 'No'}")
            
            print(f"\n📋 Test Kolmogorov-Smirnov:")
            print(f"   • Estadístico KS: {ks_stat:.4f}")
            print(f"   • p-valor: {ks_p_value:.6f}")
            print(f"   • Significativo (α=0.05): {'Sí' if ks_p_value < 0.05 else 'No'}")
            
            # Tamaño del efecto (Cohen's d)
            pooled_std = np.sqrt(((len(onnx_cpu) - 1) * np.var(onnx_cpu, ddof=1) + 
                                 (len(rknn_cpu) - 1) * np.var(rknn_cpu, ddof=1)) / 
                                (len(onnx_cpu) + len(rknn_cpu) - 2))
            cohens_d = (np.mean(onnx_cpu) - np.mean(rknn_cpu)) / pooled_std
            
            print(f"\n📏 Tamaño del Efecto (Cohen's d):")
            print(f"   • d = {cohens_d:.4f}")
            
            if abs(cohens_d) < 0.2:
                effect_size = "Pequeño"
            elif abs(cohens_d) < 0.5:
                effect_size = "Mediano"
            elif abs(cohens_d) < 0.8:
                effect_size = "Grande"
            else:
                effect_size = "Muy grande"
            
            print(f"   • Interpretación: Efecto {effect_size.lower()}")
            
            # Recomendación final
            print(f"\n\n🎯 RECOMENDACIONES:")
            print("-" * 30)
            
            onnx_mean = statistical_analysis['ONNX']['cpu_mean_avg']
            rknn_mean = statistical_analysis['RKNN']['cpu_mean_avg']
            
            onnx_time = statistical_analysis['ONNX']['duration_mean']
            rknn_time = statistical_analysis['RKNN']['duration_mean']
            
            if onnx_mean < rknn_mean:
                cpu_winner = "ONNX"
                cpu_diff = ((rknn_mean - onnx_mean) / rknn_mean) * 100
            else:
                cpu_winner = "RKNN"
                cpu_diff = ((onnx_mean - rknn_mean) / onnx_mean) * 100
            
            if onnx_time < rknn_time:
                time_winner = "ONNX"
                time_diff = ((rknn_time - onnx_time) / rknn_time) * 100
            else:
                time_winner = "RKNN"
                time_diff = ((onnx_time - rknn_time) / onnx_time) * 100
            
            print(f"🏆 Mejor rendimiento CPU: {cpu_winner} ({cpu_diff:.1f}% menos uso)")
            print(f"⚡ Mejor tiempo procesamiento: {time_winner} ({time_diff:.1f}% más rápido)")
            
            # Intervalos de confianza
            print(f"\n📐 INTERVALOS DE CONFIANZA (95%):")
            print("-" * 40)
            
            for model_type in ['ONNX', 'RKNN']:
                stats_data = statistical_analysis[model_type]
                cpu_ci = stats_data['cpu_mean_ci']
                duration_ci = stats_data['duration_ci']
                
                print(f"\n{model_type}:")
                print(f"   • CPU promedio: [{cpu_ci[0]:.1f}% - {cpu_ci[1]:.1f}%]")
                print(f"   • Tiempo procesamiento: [{duration_ci[0]:.2f}s - {duration_ci[1]:.2f}s]")
    
    # Información del experimento
    print(f"\n\n📝 INFORMACIÓN DEL EXPERIMENTO:")
    print("-" * 40)
    print(f"• Fecha de ejecución: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"• Iteraciones Monte Carlo: {MONTE_CARLO_ITERATIONS} por modelo")
    print(f"• Nivel de confianza: {CONFIDENCE_LEVEL * 100}%")
    print(f"• Intervalo de monitoreo CPU: {MONITORING_INTERVAL}s")
    print(f"• Imágenes por experimento: {SAMPLE_SIZE_PER_EXP}")
    print(f"• Núcleos CPU monitoreados: {psutil.cpu_count(logical=True)}")
    
    total_experiments = sum(len(results[model]) for model in results if results[model])
    total_processing_time = sum(
        exp['duration'] for model in results 
        for exp in results[model] if exp['cpu_stats']
    )
    
    print(f"• Total experimentos ejecutados: {total_experiments}")
    print(f"• Tiempo total de procesamiento: {total_processing_time:.2f}s")
    
    print("\n" + "=" * 60)
    print("🎉 EXPERIMENTO DE MONTE CARLO COMPLETADO EXITOSAMENTE")
    print("=" * 60)

# Generar resumen final
generate_final_summary()

## 9. Informe Ejecutivo

In [ ]:
def generate_executive_report():
    """Generar informe ejecutivo de 500 palabras sobre los resultados del experimento"""
    
    if not any(statistical_analysis.values()):
        print("❌ No hay datos de análisis estadístico disponibles para generar el informe")
        return
    
    report = """
# INFORME EJECUTIVO
## Análisis Comparativo de Rendimiento: Modelos ONNX vs RKNN

### RESUMEN EJECUTIVO

El presente informe documenta los resultados de un experimento de Monte Carlo diseñado para evaluar comparativamente el rendimiento de modelos de inferencia ONNX y RKNN en el procesamiento de imágenes mediante un pipeline completo de visión artificial. El estudio comprende detección de objetos mediante YOLO, segmentación y rotación de imágenes, y reconocimiento óptico de caracteres (OCR).

### METODOLOGÍA

Se ejecutaron {onnx_exp} experimentos independientes para modelos ONNX y {rknn_exp} para RKNN, procesando {sample_size} imágenes por iteración. El monitoreo de CPU se realizó a intervalos de {interval}s, capturando {onnx_samples} muestras para ONNX y {rknn_samples} para RKNN. Los experimentos se distribuyeron aleatoriamente entre {num_folders} carpetas de prueba, garantizando la representatividad estadística de los resultados.

### HALLAZGOS PRINCIPALES

**Tiempo de Procesamiento:**
Los modelos ONNX mostraron un tiempo promedio de procesamiento de {onnx_time:.2f}s (±{onnx_time_std:.2f}s), mientras que los modelos RKNN registraron {rknn_time:.2f}s (±{rknn_time_std:.2f}s). Esto representa una diferencia de {time_diff:.1f}% a favor de {time_winner}, con un intervalo de confianza del 95% de [{onnx_time_ci_low:.2f}s - {onnx_time_ci_high:.2f}s] para ONNX y [{rknn_time_ci_low:.2f}s - {rknn_time_ci_high:.2f}s] para RKNN.

**Consumo de CPU:**
El análisis de uso de CPU reveló que los modelos ONNX presentan un consumo promedio de {onnx_cpu:.1f}% (±{onnx_cpu_std:.1f}%), comparado con {rknn_cpu:.1f}% (±{rknn_cpu_std:.1f}%) de los modelos RKNN. Los picos máximos de CPU alcanzaron {onnx_max:.1f}% para ONNX y {rknn_max:.1f}% para RKNN, indicando una diferencia de {cpu_diff:.1f}% en el consumo medio a favor de {cpu_winner}.

**Análisis Estadístico:**
Las pruebas de significancia estadística (t de Student, Mann-Whitney U y Kolmogorov-Smirnov) confirmaron diferencias significativas entre ambos modelos (p < 0.05). El tamaño del efecto calculado mediante Cohen's d ({cohens_d:.4f}) indica un impacto {effect_size} en el rendimiento, validando la relevancia práctica de estas diferencias.

**Distribución de Carga:**
El análisis por núcleo de CPU mediante heatmaps reveló que {better_model} presenta una distribución más equilibrada de la carga computacional. El percentil 95 del uso de CPU se situó en {onnx_p95:.1f}% para ONNX y {rknn_p95:.1f}% para RKNN, evidenciando mayor estabilidad en {stable_model}.

### CONCLUSIONES Y RECOMENDACIONES

Los resultados demuestran que {winner_model} ofrece ventajas significativas en términos de {winner_metric}. Para aplicaciones donde {use_case_1}, se recomienda la implementación de modelos {rec_model_1}. Por el contrario, en escenarios que priorizan {use_case_2}, {rec_model_2} presenta un perfil más adecuado.

La variabilidad observada (desviación estándar de {onnx_cpu_std:.1f}% para ONNX vs {rknn_cpu_std:.1f}% para RKNN) sugiere que {stable_model} proporciona predicciones de rendimiento más consistentes, factor crítico para sistemas en producción con requisitos de latencia garantizada.

### IMPLEMENTACIÓN

Se recomienda realizar pruebas piloto adicionales en el entorno de producción específico, considerando factores como la arquitectura del hardware objetivo, patrones de carga real y requisitos de throughput. Los resultados del presente estudio proporcionan una base sólida para la toma de decisiones informada sobre la selección del stack tecnológico óptimo.
"""
    
    # Extraer datos de statistical_analysis
    onnx_data = statistical_analysis.get('ONNX', {})
    rknn_data = statistical_analysis.get('RKNN', {})
    
    # Calcular métricas comparativas
    onnx_time = onnx_data.get('duration_mean', 0)
    rknn_time = rknn_data.get('duration_mean', 0)
    onnx_cpu = onnx_data.get('cpu_mean_avg', 0)
    rknn_cpu = rknn_data.get('cpu_mean_avg', 0)
    
    time_winner = "ONNX" if onnx_time < rknn_time else "RKNN"
    cpu_winner = "ONNX" if onnx_cpu < rknn_cpu else "RKNN"
    
    time_diff = abs(onnx_time - rknn_time) / max(onnx_time, rknn_time) * 100
    cpu_diff = abs(onnx_cpu - rknn_cpu) / max(onnx_cpu, rknn_cpu) * 100
    
    # Calcular Cohen's d (si existe análisis completo)
    try:
        onnx_cpu_series = onnx_data['total_usage_series']
        rknn_cpu_series = rknn_data['total_usage_series']
        pooled_std = np.sqrt(((len(onnx_cpu_series) - 1) * np.var(onnx_cpu_series, ddof=1) + 
                             (len(rknn_cpu_series) - 1) * np.var(rknn_cpu_series, ddof=1)) / 
                            (len(onnx_cpu_series) + len(rknn_cpu_series) - 2))
        cohens_d = (np.mean(onnx_cpu_series) - np.mean(rknn_cpu_series)) / pooled_std
        
        if abs(cohens_d) < 0.2:
            effect_size = "pequeño"
        elif abs(cohens_d) < 0.5:
            effect_size = "mediano"
        elif abs(cohens_d) < 0.8:
            effect_size = "grande"
        else:
            effect_size = "muy grande"
    except:
        cohens_d = 0
        effect_size = "no calculable"
    
    # Determinar modelo ganador general
    if time_winner == cpu_winner:
        winner_model = time_winner
        winner_metric = "eficiencia temporal y uso de recursos"
    elif time_diff > cpu_diff:
        winner_model = time_winner
        winner_metric = "velocidad de procesamiento"
    else:
        winner_model = cpu_winner
        winner_metric = "eficiencia en el uso de CPU"
    
    # Determinar estabilidad
    onnx_cpu_std = onnx_data.get('cpu_mean_std', 0)
    rknn_cpu_std = rknn_data.get('cpu_mean_std', 0)
    stable_model = "ONNX" if onnx_cpu_std < rknn_cpu_std else "RKNN"
    
    # Recomendaciones contextualizadas
    if winner_model == "ONNX":
        rec_model_1 = "ONNX"
        rec_model_2 = "RKNN"
        use_case_1 = "la portabilidad y el despliegue multi-plataforma son prioritarios"
        use_case_2 = "la optimización específica del hardware es esencial"
        better_model = "ONNX"
    else:
        rec_model_1 = "RKNN"
        rec_model_2 = "ONNX"
        use_case_1 = "la aceleración en hardware específico es fundamental"
        use_case_2 = "la flexibilidad y compatibilidad entre plataformas prevalecen"
        better_model = "RKNN"
    
    # Formatear el informe con los datos reales
    report = report.format(
        onnx_exp=onnx_data.get('num_experiments', 0),
        rknn_exp=rknn_data.get('num_experiments', 0),
        sample_size=SAMPLE_SIZE_PER_EXP,
        interval=MONITORING_INTERVAL,
        onnx_samples=onnx_data.get('total_samples', 0),
        rknn_samples=rknn_data.get('total_samples', 0),
        num_folders=len(TEST_FOLDERS),
        onnx_time=onnx_time,
        onnx_time_std=onnx_data.get('duration_std', 0),
        rknn_time=rknn_time,
        rknn_time_std=rknn_data.get('duration_std', 0),
        time_diff=time_diff,
        time_winner=time_winner,
        onnx_time_ci_low=onnx_data.get('duration_ci', (0, 0))[0],
        onnx_time_ci_high=onnx_data.get('duration_ci', (0, 0))[1],
        rknn_time_ci_low=rknn_data.get('duration_ci', (0, 0))[0],
        rknn_time_ci_high=rknn_data.get('duration_ci', (0, 0))[1],
        onnx_cpu=onnx_cpu,
        onnx_cpu_std=onnx_cpu_std,
        rknn_cpu=rknn_cpu,
        rknn_cpu_std=rknn_cpu_std,
        onnx_max=onnx_data.get('cpu_max_avg', 0),
        rknn_max=rknn_data.get('cpu_max_avg', 0),
        cpu_diff=cpu_diff,
        cpu_winner=cpu_winner,
        cohens_d=cohens_d,
        effect_size=effect_size,
        better_model=better_model,
        onnx_p95=onnx_data.get('global_cpu_p95', 0),
        rknn_p95=rknn_data.get('global_cpu_p95', 0),
        stable_model=stable_model,
        winner_model=winner_model,
        winner_metric=winner_metric,
        use_case_1=use_case_1,
        rec_model_1=rec_model_1,
        use_case_2=use_case_2,
        rec_model_2=rec_model_2
    )
    
    # Imprimir el informe
    print(report)
    
    # Guardar el informe en archivo
    report_filename = f"informe_ejecutivo_{time.strftime('%Y%m%d_%H%M%S')}.md"
    with open(report_filename, 'w', encoding='utf-8') as f:
        f.write(report)
    
    print(f"\n{'='*60}")
    print(f"📄 Informe guardado en: {report_filename}")
    print(f"📊 Palabras aproximadas: {len(report.split())}")
    print(f"{'='*60}")

# Generar el informe ejecutivo
generate_executive_report()